# Problem Set 2: the W mass and the Jacobian peak
Generate **1,000 events per sample** in FindingZ, using the Standard Model,
LHC Run 2 and **MadGraph-only** mode. No Pythia or Delphes is needed.
This notebook reads the saved `.lhe` or `.lhe.gz` event files directly;
it does not require changes to FindingZ.

Use these process definitions in the generation card (or the corresponding course presets):

**W sample**
```
generate p p > w+, w+ > e+ ve
add process p p > w-, w- > e- ve~
```
**W + jet sample**
```
generate p p > w+ j, w+ > e+ ve
add process p p > w- j, w- > e- ve~
```
For the second sample set `ptj = 30` GeV in the run card. Keep other cuts the same;
record the actual lepton cuts, W mass and width in your saved cards. Use fresh runs.
Here `j` is an outgoing parton, not a reconstructed detector jet.

First do the short derivation in the problem set. Then implement one function below,
make the plots and interpret them. The file-loading and plotting code is supplied.
The neutrino's full momentum is used only to check the true invariant mass;
the transverse-mass calculation must use transverse components alone.

In [ ]:
from pathlib import Path
import os, gzip
import xml.etree.ElementTree as ET
import numpy as np
import matplotlib.pyplot as plt

MW = 80.4  # GeV: replace with MASS 24 in your saved param_card.dat
# Optional: list event files under FindingZ's configured run directory.
run_root = os.environ.get('FINDINGZ_RUN_ROOT')
if run_root:
    for path in sorted(Path(run_root).rglob('*.lhe*')):
        print(path)

## Load the two samples
Enter the actual saved event-file paths below. FindingZ's run directory contains
`events.lhe` or `events.lhe.gz`; you can also download the LHE files and use local paths.
Only numpy and matplotlib are needed beyond Python's standard library.
The loader checks for one electron/positron and one electron neutrino/antineutrino
per event and equal positive event weights (the unweighted LO samples requested here).

In [ ]:
W_FILE = Path('PATH_TO_W_EVENTS.lhe.gz')
WJ_FILE = Path('PATH_TO_W_JET_EVENTS.lhe.gz')

def read_sample(path):
    electrons, neutrinos, jet_pts, weights = [], [], [], []
    opener = gzip.open if path.suffix == '.gz' else open
    with opener(path, 'rt') as stream:
        for _, element in ET.iterparse(stream, events=('end',)):
            if element.tag != 'event':
                continue
            lines = [line.strip() for line in (element.text or '').splitlines()
                     if line.strip() and not line.lstrip().startswith('#')]
            header = lines[0].split()
            n = int(header[0])
            number = lambda x: float(x.replace('D', 'E').replace('d', 'e'))
            weights.append(number(header[2]))
            final = []
            for line in lines[1:n+1]:
                fields = line.split()
                if int(fields[1]) == 1:
                    # Four-vector order is (E, px, py, pz).
                    final.append((int(fields[0]), np.array([
                        number(fields[9]), number(fields[6]),
                        number(fields[7]), number(fields[8])])))
            e = [p for pid, p in final if abs(pid) == 11]
            nu = [p for pid, p in final if abs(pid) == 12]
            if len(e) != 1 or len(nu) != 1:
                raise ValueError('Choose a W -> e nu sample with exactly one e and one nu per event.')
            electrons.append(e[0]); neutrinos.append(nu[0])
            jet_pts.append(max([np.hypot(p[1], p[2]) for pid, p in final
                                if abs(pid) in {1,2,3,4,5,21}], default=0.))
            element.clear()
    if not electrons:
        raise ValueError(f'No events found in {path}')
    weights = np.array(weights)
    if not np.all(weights > 0) or not np.allclose(weights / weights[0], 1.):
        raise ValueError('Use equal-positive-weight LO events; these plots do not implement reweighting.')
    e, nu = np.array(electrons), np.array(neutrinos)
    pair = e + nu
    mass2 = pair[:,0]**2 - np.sum(pair[:,1:]**2, axis=1)
    sample = dict(pt=np.hypot(e[:,1],e[:,2]), phi=np.arctan2(e[:,2],e[:,1]),
                  npt=np.hypot(nu[:,1],nu[:,2]), nphi=np.arctan2(nu[:,2],nu[:,1]),
                  mass=np.sqrt(np.maximum(mass2,0.)),
                  recoil=np.hypot(pair[:,1],pair[:,2]), jet_pt=np.array(jet_pts))
    print(path.name, ':', len(e), 'events')
    if len(e) != 1000:
        print('Expected 1,000 events: check the generation settings.')
    return sample

samples = {'W': read_sample(W_FILE), 'W + jet': read_sample(WJ_FILE)}
print('Largest W-only recoil [GeV]:', samples['W']['recoil'].max())
print('Smallest W+jet parton pT [GeV]:', samples['W + jet']['jet_pt'].min())

## 1. Implement transverse mass
Use your expression from part (a). `pt, phi` are the electron's transverse
momentum magnitude and azimuth; `npt, nphi` are the corresponding neutrino quantities.
Replace the `raise` statement with your formula. The checks below test simple geometries.

In [ ]:
def transverse_mass(pt, npt, phi, nphi):
    # TODO: return m_T using only these transverse quantities.
    raise NotImplementedError('Insert your transverse-mass formula from part (a).')

np.testing.assert_allclose(transverse_mass(40.,40.,0.,0.), 0., atol=1e-10)
np.testing.assert_allclose(transverse_mass(40.,40.,0.,np.pi), 80.)
np.testing.assert_allclose(transverse_mass(30.,40.,0.,np.pi/2), np.sqrt(2400.))
for sample in samples.values():
    sample['mt'] = transverse_mass(sample['pt'], sample['npt'], sample['phi'], sample['nphi'])

## 2. The Jacobian peak
For a fixed-mass W with zero transverse recoil and an isotropic reference decay,
\[
\frac{dP}{dp_T}=\frac{4p_T}{m_W^2\sqrt{1-4p_T^2/m_W^2}}.
\]
Derive this result, including both angular solutions. Why is the divergence integrable?
The code integrates this reference over each bin instead of evaluating it at its singular endpoint.
Compare its shape with the generated W sample. Physical polarization, finite width and generation
cuts can change the comparison: an isotropic reference is not the exact W production prediction.

In [ ]:
def shape(ax, values, edges, label):
    counts, _ = np.histogram(values, edges)
    ax.stairs(counts / (len(values)*np.diff(edges)), edges, label=label)
    overflow = np.count_nonzero((values < edges[0]) | (values > edges[-1]))
    if overflow: print(label, ':', overflow, 'events outside displayed range')

def ideal_bin_density(edges, endpoint):
    cdf = 1 - np.sqrt(1 - np.clip(edges / endpoint, 0., 1.)**2)
    return np.diff(cdf) / np.diff(edges)

edges = np.arange(0., max(102., 2*MW), 2.)
fig, ax = plt.subplots(figsize=(7,4), constrained_layout=True)
shape(ax, samples['W']['pt'], edges, 'W sample')
ax.stairs(ideal_bin_density(edges, MW/2), edges, label='Isotropic, fixed mass, no cuts')
ax.axvline(MW/2, color='black', ls='--', label=r'$m_W/2$')
ax.set(xlabel=r'$p_{Te}$ [GeV]', ylabel='Probability density [1/GeV]')
ax.legend(); plt.show()

## 3. Compare mass estimators and turn on recoil
Plot all three quantities for each sample, using the same bins. Estimate the mass
from the W-only distributions. Then explain what changes when a jet carries recoil.
Each histogram is normalized to its full sample size: these are shape comparisons,
not cross-section comparisons. Record any generation cuts that distort the shapes.

In [ ]:
edges = np.arange(0., max(202., 3*MW), 2.)
fig, axes = plt.subplots(1,2,figsize=(12,4),sharex=True,sharey=True,constrained_layout=True)
for ax, (label, sample) in zip(axes, samples.items()):
    for values, name in [(2*sample['pt'],r'$2p_{Te}$'), (sample['mt'],r'$m_T$'),
                         (sample['mass'],r'$m_{e\nu}$ (truth)')]:
        shape(ax, values, edges, name)
    ax.axvline(MW,color='black',ls='--',label='Nominal W mass')
    ax.set(title=label,xlabel='Mass estimator [GeV]',ylabel='Probability density [1/GeV]')
    ax.legend()
plt.show()

# Compare against the ACTUAL pair mass, not the nominal mass parameter.
for label, sample in samples.items():
    tolerance = 1e-6 * np.maximum(sample['mass'],1.)
    print(label)
    for name, values in [('2 pTe',2*sample['pt']),('mT',sample['mt'])]:
        print('  Fraction with',name,'> true mass:',
              np.mean(values > sample['mass'] + tolerance))
    print('  Largest |mT - 2 pTe| [GeV]:',np.max(np.abs(sample['mt']-2*sample['pt'])))

### Your conclusions
- Explain the Jacobian peak using your change of variables, not just the histogram.
- Why does zero transverse recoil imply $m_T=2p_{Te}$? Does a longitudinal boost change this?
- Which estimator remains a lower bound on $m_{e\nu}$ after adding recoil? Explain event by event.
- Why can $m_T$ exceed the nominal $m_W$ without violating this bound?

These are generator-level events. There is no detector smearing in these plots.
The full neutrino momentum provides a truth check; it would not be known in a collider experiment.